#<h1 align="center">**LMF Interactive Scatterplot**</h1>




<div align="justify">

This is an interactive scatter plot where you can easily explore all the processed samples from the Low Methane Forages project.

First go to **File** → **"Save a copy in Drive"**. This will create a copy of the notebook in your Google Drive. You can then edit the notebook and explore the data using the interactive controls.

To activate the visualization options, click the run_button.jpg button located at the top of the panel. Then, explore different combinations of X and Y axes, subsets, and functional groups. Once you have adjusted the plot to your preference, you can export it by clicking the three_dots_button.jpg button in the upper-right corner of the scatter plot.

**Advanced options**: You can customize the benchmark visualization by adding one or more id_lab values to the list in the second cell of the **Plot code** section. Benchmark samples will be highlighted in **purple**.

To highlight one or more samples of interest, add their id_lab values to the target list. These samples will be displayed in **red**.

Don't know the id_lab of your sample of interest? Explore the complete dataset [here.](https://github.com/maurope/lmf/blob/main/output/2026_05_22_data_delivery/08_dashboard_june_2026/compiled_dashboard_2026_7_10.csv)

</div>


# 1.0 Import libraries

In [7]:
import pandas as pd
import altair as alt
import ipywidgets as widgets
from IPython.display import display

#2.0  Data load

In [8]:
url = "https://raw.githubusercontent.com/maurope/lmf/main/output/2026_05_22_data_delivery/08_dashboard_june_2026/compiled_dashboard_2026_7_10.csv"

df = pd.read_csv(url)
df

,id_lab,id,subset,no,requisitioner,tax_name,functional_group,n_replicates_nutrition,dm_percentage,ash_dm,om_percentage,pc_percentage_dm,adf_percentage_dm,ndf_percentage_dm,n_replicates_gas,ch4_percentage_in_gas_8h,ch4_percentage_in_gas_24h,methane_intensity,tddm
0,F24-3470,CIAT-11194,1,199.0,Genetic_bank,Stylosanthes hamata,Herbaceous_legumes,2,92.89,9.61,90.39,21.18,30.18,55.94,6.0,15.17,18.02,46.12,59.25
1,F24-3471,CIAT-11999,1,201.0,Genetic_bank,Stylosanthes guianensis,Herbaceous_legumes,2,93.50,10.46,89.54,20.09,33.90,61.32,9.0,12.98,16.25,46.61,58.55
2,F24-3472,CIAT-12318,1,203.0,Genetic_bank,Stylosanthes hamata,Herbaceous_legumes,2,94.10,11.08,88.92,22.05,29.34,55.26,9.0,13.99,16.96,52.27,56.88
3,F24-3427,CIAT-1257,1,113.0,Genetic_bank,Stylosanthes scabra,Herbaceous_legumes,2,92.72,9.52,90.48,17.96,35.90,69.91,9.0,15.71,17.60,46.32,58.95
4,F24-3473,CIAT-13575,1,205.0,Genetic_bank,Desmodium incanum,Herbaceous_legumes,2,94.14,11.52,88.48,23.17,44.09,57.61,9.0,13.60,16.23,42.43,42.85
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
656,F25-2189,CIAT-BH-22-2365,4,113.0,Genetic_bank,Urochloa humidicola,Grass,2,97.10,12.34,87.66,9.98,31.98,64.14,5.0,14.00,14.44,66.84,39.83
657,F25-2190,CIAT-BH-22-3572,4,115.0,Genetic_bank,Urochloa humidicola,Grass,2,96.50,10.04,89.96,7.98,32.77,68.19,5.0,13.58,14.19,154.85,20.76
658,F25-2187,CIAT-BH-22-3830,4,109.0,Genetic_bank,Urochloa humidicola,Grass,2,95.96,10.19,89.81,8.04,32.62,67.43,6.0,13.90,14.78,75.10,40.80
659,F25-2640,Cayman-Br-02-1752,4,61.0,Isabel Molina,Urochloa interespecific,Grass,2,97.00,9.65,90.35,8.05,26.24,62.13,8.0,12.99,13.37,50.17,50.22


# 3.0 Plot code

In [9]:
df = df.copy()
df["subset"] = df["subset"].astype(str)
df["functional_group"] = df["functional_group"].fillna("Unknown")

In [16]:
# ---------------------------------------------
# Lists of special samples
# ---------------------------------------------

target = ["F25-2079"]
star_grass_control_ids = ["F25-0008"]
benchmark_ids = ["F25-2136","F25-2137","F25-2138","F25-2139","F25-2140","F25-2141","F25-2566","F25-2567","F25-2568", "F25-2624", "F25-2626", "F25-2627", "F25-2628"]

In [17]:
# ============================================================
# Widgets
# ============================================================

numeric_columns = sorted(df.select_dtypes(include="number").columns.tolist())

# default values
default_x = "tddm"
default_y = "methane_intensity"

# if any column does not exist, use the first
if default_x not in numeric_columns:
    default_x = numeric_columns[0]

if default_y not in numeric_columns:
    default_y = numeric_columns[1] if len(numeric_columns) > 1 else numeric_columns[0]

x_widget = widgets.Dropdown(
    options=numeric_columns,
    value=default_x,
    description="X:"
)

y_widget = widgets.Dropdown(
    options=numeric_columns,
    value=default_y,
    description="Y:"
)

subset_widget = widgets.Dropdown(
    options=["All"] + sorted(df["subset"].astype(str).unique().tolist()),
    value="All",
    description="Subset:"
)

functional_widget = widgets.Dropdown(
    options=["All"] + sorted(df["functional_group"].dropna().unique().tolist()),
    value="All",
    description="Group:"
)

# ============================================================
# Plot function
# ============================================================

def plot(x, y, subset, functional_group):

    data = df.copy()

    # Filter by subset
    if subset != "All":
        data = data[data["subset"].astype(str) == subset]

    # Filter by functional group
    if functional_group != "All":
        data = data[data["functional_group"] == functional_group]

    # Create plotting groups
    data["plot_group"] = data["functional_group"]

    data.loc[
        data["id_lab"].isin(benchmark_ids),
        "plot_group"
    ] = "Benchmark"

    data.loc[
        data["id_lab"].isin(star_grass_control_ids),
        "plot_group"
    ] = "Star Grass Control"

    data.loc[
        data["id_lab"].isin(target),
        "plot_group"
    ] = "Target"

    # Scatter plot
    chart = (
        alt.Chart(data)
        .mark_circle(size=70)
        .encode(
            x=alt.X(
                x,
                title=x.replace("_", " ").title()
            ),
            y=alt.Y(
                y,
                title=y.replace("_", " ").title()
            ),
            color=alt.Color(
                "plot_group:N",
                scale=alt.Scale(
                    domain=[
                        "Herbaceous_legumes",
                        "Grass",
                        "others",
                        "Shrub_Trees",
                        "Benchmark",
                        "Star Grass Control",
                        "Target",
                    ],
                    range=[
                        "green",
                        "cornflowerblue",
                        "gray",
                        "orange",
                        "purple",
                        "black",
                        "red",
                    ],
                ),
                legend=alt.Legend(title="Sample Type"),
            ),
            tooltip=[
                "id_lab",
                "id",
                "tax_name",
                "functional_group",
                "plot_group",
                "subset",
                "n_replicates_nutrition",
                "dm_percentage",
                "ash_dm",
                "om_percentage",
                "pc_percentage_dm",
                "adf_percentage_dm",
                "ndf_percentage_dm",
                "n_replicates_gas",
                "ch4_percentage_in_gas_8h",
                "ch4_percentage_in_gas_24h",
                "methane_intensity",
                "tddm",
            ],
        )
        .properties(
            width=850,
            height=600,
            title=f"{y.replace('_',' ').title()} vs {x.replace('_',' ').title()}",
        )
        .configure_axis(
            labelFontSize=14,
            titleFontSize=18,
        )
        .configure_legend(
            titleFontSize=16,
            labelFontSize=14,
        )
        .configure_title(
            fontSize=20,
        )
        .interactive()
    )

    display(chart)

# ============================================================
# Interactive dashboard
# ============================================================

controls = widgets.VBox([
    widgets.HBox([x_widget, y_widget]),
    widgets.HBox([subset_widget, functional_widget]),
])

output = widgets.interactive_output(
    plot,
    {
        "x": x_widget,
        "y": y_widget,
        "subset": subset_widget,
        "functional_group": functional_widget,
    },
)


#4.0  Interactive dashboard

In [18]:
display(controls, output)

Output()